# Figure. Target Employment Measures in Maricopa County, AZ

## Purpose
Creates the manuscript-ready target employment measures table using the current master occupation file.

## Source
`Master_CIP_SOC_Matched_Output_3.xlsx` → `Master_Matched_Data`

This workbook contains the approved occupation universe, 2025 employment, education level, projected annual openings, projected annual job change, wages, star rating, industry, Semiconductor flag, and CIP-to-SOC relationships.

## Sector definitions
Results always appear in this order:

1. Healthcare and Social Assistance
2. Manufacturing
3. Construction
4. Semiconductors

Semiconductors are identified with `Semiconductor Flag = Yes` and are a subset of Manufacturing.

## Outputs
The Excel workbook contains:
- **All Occupations Table**
- **5-Star Only Table**
- **All Occupations Detail**
- **5-Star Occupations Detail**
- **5-Star Semiconductors**

The Location Quotient row is included as a clearly labeled placeholder until the final LQ values are supplied.


In [1]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ============================================================
# COMPUTER — CHANGE ONLY THIS VALUE
# ============================================================
# "W" = work computer
# "P" = personal computer
# "S" = Shared Drive source
COMPUTER = "W"

MASTER_PATHS = {
    "W": Path(
        r"C:\Users\301533\Documents\Walmart\IPEDS\Files"
        r"\Master_CIP_SOC_Matched_Output_3.xlsx"
    ),
    "P": Path(
        r"C:\Users\Arielle\Desktop\Desktop\New folder\Walmart"
        r"\Master_CIP_SOC_Matched_Output_3.xlsx"
    ),
    "S": Path(
        r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
        r"\SOC NAICS List\Economic Analysis\Walmart_LMI_Analysis\Current\Python"
        r"\Master_CIP_SOC_Matched_Output_3.xlsx"
    ),
}

BASE_FOLDERS = {
    "W": Path(r"C:\Users\301533\Documents\Walmart\IPEDS\Files"),
    "P": Path(r"C:\Users\Arielle\Desktop\Desktop\New folder\Walmart"),
    "S": Path(
        r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
        r"\SOC NAICS List\Economic Analysis\Walmart_LMI_Analysis\Current\Python"
    ),
}

OUTPUT_FOLDER = Path(
    r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
    r"\SOC NAICS List\Excel Sheets\Arielle Tables"
)

key = str(COMPUTER).strip().upper()
if key not in MASTER_PATHS:
    raise ValueError('COMPUTER must be "W", "P", or "S".')

MASTER_FILE = MASTER_PATHS[key]
BASE_FOLDER = BASE_FOLDERS[key]

SECTOR_ORDER = [
    "Healthcare and Social Assistance",
    "Manufacturing",
    "Construction",
    "Semiconductors",
]

FIVE_STAR = 5

def clean_soc(value):
    if pd.isna(value):
        return np.nan
    digits = re.sub(r"\D", "", str(value))
    if len(digits) >= 6:
        return f"{digits[:2]}-{digits[2:6]}"
    return np.nan

def clean_cip(value):
    if pd.isna(value):
        return np.nan
    is_numeric_input = isinstance(value, (int, float, np.integer, np.floating))
    s = str(value).strip().replace('="', '').replace('"', '').replace("'", "")
    s = re.sub(r"[^0-9.]", "", s)
    if not s:
        return np.nan
    if "." in s:
        left, right = s.split(".", 1)
        right = re.sub(r"\D", "", right)
        if is_numeric_input and 1 <= len(right) <= 4:
            return f"{left.zfill(2)}.{right.ljust(4, '0')[:4]}"
        if len(right) >= 4:
            return f"{left.zfill(2)}.{right[:4]}"
    digits = re.sub(r"\D", "", s)
    if len(digits) == 6:
        return f"{digits[:2]}.{digits[2:]}"
    return np.nan

def to_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

def load_master():
    if not MASTER_FILE.exists():
        raise FileNotFoundError(
            f"Master file not found:\n{MASTER_FILE}\n\n"
            "Change COMPUTER or update MASTER_PATHS."
        )

    master = pd.read_excel(
        MASTER_FILE,
        sheet_name="Master_Matched_Data"
    ).copy()

    required = [
        "SOC_Code",
        "Job Title",
        "Employment ('25)",
        "Education Level",
        "Projected Ann. Job Openings ('25 - '27)",
        "Projected Ann. Job Chg. ('25-'27)",
        "Average Wage ('25)",
        "Sum of Occupation Rating by Ed Lvl1",
        "Industry",
        "Semiconductor Flag",
        "CIP_Code",
        "CIP_Title",
    ]

    missing = [c for c in required if c not in master.columns]
    if missing:
        raise KeyError(f"Master workbook is missing required columns: {missing}")

    master["soc"] = master["SOC_Code"].map(clean_soc)
    master["cip_current"] = master["CIP_Code"].map(clean_cip)
    master["stars"] = pd.to_numeric(
        master["Sum of Occupation Rating by Ed Lvl1"],
        errors="coerce"
    )
    master["semiconductor_yes"] = (
        master["Semiconductor Flag"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("yes")
    )

    return master

def sectorize(master, five_star_only=True, require_cip=False):
    """
    Return the occupation/CIP universe in the manuscript sector order.

    Manufacturing includes the full Manufacturing universe.
    Semiconductors are also shown separately as a flagged subset of Manufacturing.
    """

    x = master.copy()

    if five_star_only:
        x = x[x["stars"].eq(FIVE_STAR)].copy()

    x = x[x["soc"].notna()].copy()

    if require_cip:
        x = x[x["cip_current"].notna()].copy()

    pieces = []

    definitions = [
        (
            "Healthcare and Social Assistance",
            x["Industry"].eq("Healthcare")
        ),
        (
            "Manufacturing",
            x["Industry"].eq("Manufacturing")
        ),
        (
            "Construction",
            x["Industry"].eq("Construction")
        ),
        (
            "Semiconductors",
            x["semiconductor_yes"]
        ),
    ]

    for sector, mask in definitions:
        part = x.loc[mask].copy()
        part["sector"] = sector
        pieces.append(part)

    result = pd.concat(pieces, ignore_index=True)

    result["sector"] = pd.Categorical(
        result["sector"],
        categories=SECTOR_ORDER,
        ordered=True
    )

    return result.sort_values(["sector", "soc"]).reset_index(drop=True)

master = load_master()

five_star_universe = sectorize(
    master,
    five_star_only=True,
    require_cip=False
)

counts = (
    five_star_universe
    .drop_duplicates(["sector", "soc"])
    .groupby("sector", observed=False)["soc"]
    .nunique()
    .reindex(SECTOR_ORDER)
)

print("Master:", MASTER_FILE)
print("\nFive-star occupation counts:")
print(counts)

semi_count = int(counts.get("Semiconductors", 0))
if semi_count != 4:
    raise ValueError(
        f"Expected 4 five-star Semiconductor occupations; found {semi_count}. "
        "Review Semiconductor Flag before continuing."
    )


Master: C:\Users\301533\Documents\Walmart\IPEDS\Files\Master_CIP_SOC_Matched_Output_3.xlsx

Five-star occupation counts:
sector
Healthcare and Social Assistance    25
Manufacturing                       11
Construction                         9
Semiconductors                       4
Name: soc, dtype: int64


In [2]:
# ============================================================
# BUILD ALL-OCCUPATION AND 5-STAR OCCUPATION UNIVERSES
# ============================================================

all_universe = sectorize(
    master,
    five_star_only=False,
    require_cip=False
)

five_star_universe = sectorize(
    master,
    five_star_only=True,
    require_cip=False
)

def occupation_level(df):
    """One record per sector/SOC before calculating sector totals."""
    out = df.copy()

    out["employment"] = to_number(out["Employment ('25)"])
    out["average_wage"] = to_number(out["Average Wage ('25)"])
    out["annual_openings"] = to_number(
        out["Projected Ann. Job Openings ('25 - '27)"]
    )
    out["annual_growth_rate"] = (
        to_number(out["Projected Ann. Job Chg. ('25-'27)"]) / 100
    )

    return (
        out.sort_values(["sector", "soc"])
        .drop_duplicates(["sector", "soc"])
        .copy()
    )

all_occ = occupation_level(all_universe)
five_occ = occupation_level(five_star_universe)

print("All occupations:")
print(
    all_occ.groupby("sector", observed=False)["soc"]
    .nunique()
    .reindex(SECTOR_ORDER)
)

print("\n5-star occupations:")
print(
    five_occ.groupby("sector", observed=False)["soc"]
    .nunique()
    .reindex(SECTOR_ORDER)
)


All occupations:
sector
Healthcare and Social Assistance    51
Manufacturing                       70
Construction                        35
Semiconductors                      12
Name: soc, dtype: int64

5-star occupations:
sector
Healthcare and Social Assistance    25
Manufacturing                       11
Construction                         9
Semiconductors                       4
Name: soc, dtype: int64


In [3]:
# ============================================================
# CALCULATE TABLE MEASURES
# ============================================================

LQ_VALUES = {
    "Healthcare and Social Assistance": None,
    "Manufacturing": None,
    "Construction": None,
    "Semiconductors": None,
}

def summarize_sector(df):
    employment = df["employment"].sum(min_count=1)
    wage = df["average_wage"].mean()
    annual_openings = df["annual_openings"].sum(min_count=1)

    # Source field is an annualized percentage change.
    # Convert it to an approximate two-year change for the manuscript row.
    growth_count = (
        df["employment"]
        * ((1 + df["annual_growth_rate"]) ** 2 - 1)
    ).sum(min_count=1)

    growth_rate = (
        growth_count / employment
        if pd.notna(employment) and employment > 0
        else np.nan
    )

    return {
        "Occupations Included": df["soc"].nunique(),
        "Employment": employment,
        "Average Wage": wage,
        "Projected Annual Openings": annual_openings,
        "Approx. 2-Year Projected Growth": growth_count,
        "Approx. 2-Year Projected Growth Rate": growth_rate,
    }

def build_table(df):
    summaries = {
        sector: summarize_sector(
            df[df["sector"].eq(sector)]
        )
        for sector in SECTOR_ORDER
    }

    measures = [
        "Occupations Included",
        "Employment",
        "Average Wage",
        "Projected Annual Openings",
        "Approx. 2-Year Projected Growth",
        "Approx. 2-Year Projected Growth Rate",
        "Employment Location Quotient",
    ]

    rows = []
    for measure in measures:
        row = {"Measure": measure}

        for sector in SECTOR_ORDER:
            if measure == "Employment Location Quotient":
                row[sector] = LQ_VALUES[sector]
            else:
                row[sector] = summaries[sector].get(measure)

        rows.append(row)

    return pd.DataFrame(rows)

all_table = build_table(all_occ)
five_star_table = build_table(five_occ)

print("ALL OCCUPATIONS")
display(all_table)

print("\n5-STAR OCCUPATIONS ONLY")
display(five_star_table)


ALL OCCUPATIONS


,Measure,Healthcare and Social Assistance,Manufacturing,Construction,Semiconductors
0,Occupations Included,51.000000,70.000000,35.000000,12.000000
1,Employment,262627.000000,148585.000000,157962.000000,26812.000000
2,Average Wage,111139.607843,66373.428571,57408.742857,98099.000000
3,Projected Annual Openings,26532.000000,14156.000000,12934.000000,1844.000000
4,Approx. 2-Year Projected Growth,10110.769938,1675.217072,1106.220317,524.613238
5,Approx. 2-Year Projected Growth Rate,0.038499,0.011274,0.007003,0.019566
6,Employment Location Quotient,NaN,NaN,NaN,NaN



5-STAR OCCUPATIONS ONLY


,Measure,Healthcare and Social Assistance,Manufacturing,Construction,Semiconductors
0,Occupations Included,25.000000,11.000000,9.000000,4.000000
1,Employment,165981.000000,60308.000000,118331.000000,16555.000000
2,Average Wage,106402.800000,95450.000000,62730.888889,118545.000000
3,Projected Annual Openings,15762.000000,5509.000000,9911.000000,1346.000000
4,Approx. 2-Year Projected Growth,9135.036447,1519.182672,1172.110326,465.621936
5,Approx. 2-Year Projected Growth Rate,0.055037,0.025190,0.009905,0.028126
6,Employment Location Quotient,NaN,NaN,NaN,NaN


In [4]:
# ============================================================
# EXPORT
# ============================================================

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_FOLDER
    / "Figure_Target_Employment_Measures_Maricopa.xlsx"
)

detail_columns = [
    "sector",
    "SOC_Code",
    "Job Title",
    "Employment ('25)",
    "Education Level",
    "Projected Ann. Job Openings ('25 - '27)",
    "Projected Ann. Job Chg. ('25-'27)",
    "Average Wage ('25)",
    "Sum of Occupation Rating by Ed Lvl1",
    "Industry",
    "Semiconductor Flag",
]

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    all_table.to_excel(
        writer,
        sheet_name="All Occupations Table",
        index=False
    )

    five_star_table.to_excel(
        writer,
        sheet_name="5-Star Only Table",
        index=False
    )

    all_occ[detail_columns].to_excel(
        writer,
        sheet_name="All Occupations Detail",
        index=False
    )

    five_occ[detail_columns].to_excel(
        writer,
        sheet_name="5-Star Occupations Detail",
        index=False
    )

    five_occ[
        five_occ["sector"].eq("Semiconductors")
    ][detail_columns].to_excel(
        writer,
        sheet_name="5-Star Semiconductors",
        index=False
    )

    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        ws.sheet_view.showGridLines = False

print("Saved:", OUTPUT_FILE)


Saved: G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis\SOC NAICS List\Excel Sheets\Arielle Tables\Figure_Target_Employment_Measures_Maricopa.xlsx
